In [8]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# Load environment variables (.env file should contain GOOGLE_API_KEY)
load_dotenv()

# Initialize Gemini Model
chat_model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest", # یا gemini-1.5-flash
    temperature=0.1
)

print("✅ Gemini model initialized successfully!")

✅ Gemini model initialized successfully!


In [9]:
# 1. Zero-Shot Prompt
prompt_without_few_shot = ChatPromptTemplate.from_messages([
    ("system", "احساس متن زیر را دسته‌بندی کن."),
    ("human", "{input}")
])

chain = prompt_without_few_shot | chat_model

print("⏳ Sending Zero-Shot request...")
response = chain.invoke({"input": "آخه کی می‌ره این فیلمو ببینه؟"})

print("\n--- Zero-Shot Response ---")
print(response.content)

⏳ Sending Zero-Shot request...

--- Zero-Shot Response ---
[{'type': 'text', 'text': 'احساس این متن **«منفی»** است و به طور خاص نشان\u200cدهنده **«انتقاد، نارضایتی و ناامیدی»** نسبت به فیلم است. \n\nلحن جمله **«محافظه\u200cکارانه/عامیانه»** و همراه با نوعی تعجب تمسخرآمیز است.', 'extras': {'signature': 'EmAKXgFpFH0TFknhSQvJRYAyd9Yid3BPd2BrCvh+6LF8cMLcEclao9GPj0UMfgsmhFiHlFGn6TH28Ch9Kaci1AIJVUm3N4c3cUBpp4FEgtthLfGG3TSkG26dN3viaLjiBl8='}}]


In [10]:
# Define examples
examples = [
    {"question": "من عاشق این فیلمم!", "answer": "مثبت"},
    {"question": "از این صندلی خوشم نمی‌اد", "answer": "منفی"},
    {"question": "بد نبود، ولی عالی هم نبود", "answer": "خنثی"}
]

# Create a prompt template for the examples
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}"),
    ("ai", "{answer}"),
])

# Create the few-shot template (Fake Chat History)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# Build the final prompt
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "احساس متن زیر را دسته‌بندی کن."),
    few_shot_prompt, # Injecting the fake chat history here
    ("human", "{question}"),
])

chain_with_few_shot = final_prompt | chat_model

print("⏳ Sending Few-Shot request...")
response = chain_with_few_shot.invoke({"question": "آخه کی می‌ره این فیلمو ببینه؟"})

print("\n--- Few-Shot Response ---")
print(response.content)

⏳ Sending Few-Shot request...

--- Few-Shot Response ---
[{'type': 'text', 'text': 'منفی (یا انتقادی/مخالف)', 'extras': {'signature': 'EmAKXgFpFH0T4kDRHQEYFQ27RGcYmQpMMgxkZ8u7a8zLmiw9Zb63kgeuYzPyEatjdzA89hYZ0rbtH4JJ0PCACNcWy51SE8Ep2e6rSP5u2QhDAe/WElWntHTqho+EnHlitlw='}}]


In [11]:
# 1. Zero-Shot Context Request
without_few_shot_context = ChatPromptTemplate.from_messages([
    ("system", "تو اطلاعات خوبی درباره شرکت کوئرا داری"),
    ("system", "answer very short"),
    ("human", "{question}")
])

chain_without_few_shot_context = without_few_shot_context | chat_model

print("⏳ Sending Zero-Shot request for Quera context...")
chat_ai = chain_without_few_shot_context.invoke({"question": "چرا «شفیعی موقت» داریم؟"})

print("\n--- Zero-Shot Response ---")
print(chat_ai.content)

⏳ Sending Zero-Shot request for Quera context...

--- Zero-Shot Response ---
[{'type': 'text', 'text': 'عبارت «شفیعی موقت» در کوئرا اشاره به یک طنز یا میم درون\u200cسازمانی (یا جامعه برنامه\u200cنویسان) دارد که معمولاً به تغییرات مکرر، نقش\u200cهای موقت یا شخصی اشاره می\u200cکند که قرار است جایگزین فردی به نام «شفیعی» شود، اما وضعیت او همیشه به حالت «موقت» باقی می\u200cماند! \n\n*(اگر منظور دیگری از این اصطلاح در رویداد یا چالش خاصی از کوئرا دارید، بفرمایید تا دقیق\u200cتر بگویم).*', 'extras': {'signature': 'EmAKXgFpFH0TR1U/KVYEvRB8oxq30PbXYZzFvjK9Vrbaq8htAhX4aX9wPQc93SAT2xn4bC7UYSU3f9FL+PKCduHhRRcbye+4pF4QDxZdLZfz7yHemd/Kh7GbN5bzWScVPb4='}}]


In [12]:
# Define external knowledge as examples
examples_context = [
    {"question": "کوئرا کجاست؟", 
     "answer":  "کوئرا جامعه ای برای برنامه نویس های ایرانه!"},
    {"question": "محصولات کوئرا شامل چه چیز‌هایی می باشد.", 
     "answer":  "سامانه کمک آموزشی برنامه نویسی، مسابقات آنلاین برنامه‌ نویسی، سامانه کاریابی برنامه نویسی، کوئرا کالج،بوت‌کمپ‌های برنامه نویسی"},
    {"question": "مدیر تیم آموزش کیست؟",
     "answer": "در زمان طراحی این دوره، مدیر تیم آموزش کوئرا علی شفیعی است. او به دلایل متعددی از جمله ملاقات با افسر ارشد ارتش اتریش به این کشور سفر کرده است"},
    {"question": "امین حسینی کیست؟",
     "answer": "مدیر تیم عمومی آموزش و «شفیعی موقت» یک"},
    {"question": "امین انوری کیست؟", 
     "answer": "مسئول مسابقات و «شفیعی موقت» دو"}
]

# Create the few-shot template for context
example_prompt_context = ChatPromptTemplate.from_messages([
    ("human", "{question}"),
    ("ai", "{answer}")
])

few_shot_prompt_context = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt_context,
    examples=examples_context,
)

# Build final context prompt
final_prompt_context = ChatPromptTemplate.from_messages([
    ("system", "تو اطلاعات خوبی درباره شرکت کوئرا داری"),
    ("system", "answer very short"),
    few_shot_prompt_context,
    ("human", "{question}")
])

chain_with_few_shot_context = final_prompt_context | chat_model

print("⏳ Sending Few-Shot request for Quera context...")
chat_ai = chain_with_few_shot_context.invoke({"question": "چرا «شفیعی موقت» داریم؟"})

print("\n--- Few-Shot Context Response ---")
print(chat_ai.content)

⏳ Sending Few-Shot request for Quera context...

--- Few-Shot Context Response ---
[{'type': 'text', 'text': 'چون مدیر اصلی آموزش (علی شفیعی) رفته اتریش!', 'extras': {'signature': 'EmAKXgFpFH0TykjpdKN2vE0yU0zwg/iM+W0THiK/IcHEAaNUueptmXYiGULdtqRLrhOLHO4+WEkOQZubGwhRIBYBu5QgaljEquu46044feUeKw+vOOUbbGMlUCvJ08gEyJ0='}}]
